# One-cell comparison from your three notebooks in My Drive

1. Keep the three uploaded `.ipynb` notebooks in **My Drive**. Their original filenames are recognised automatically.
2. Open this notebook in Google Colab and select **Runtime → Change runtime type → T4 GPU**.
3. Run the **single code cell below** and connect the Drive account containing those notebooks and your saved Attention U-Net run.
4. Keep Colab connected. If peer checkpoints are missing, it downloads about **0.9 GB** of public ISIC data and **trains the missing pipelines again**. This can take hours.
5. At the end, a ZIP downloads. Use **Slide1_Shared_Image_Comparison.png** for your slide.

**No peer `.keras` uploads are needed.** This runner rebuilds missing models from the supplied code and saves completed stages to a separate folder, so a rerun can reuse them. It still needs **your own original Attention U-Net models** in Drive to preserve the final candidate. It checks for those before training other models.

The selected image, **ISIC_0000231**, is **benign** and was the best recorded Attention segmentation example (historical Dice about **0.9876**). Its image and expert mask are embedded. This is an explicitly **selected best-case illustration**, not proof of an overall winner. The code calculates the actual current per-image scores without altering any outputs or thresholds.

The uploaded file named U-Net++ contains a standard U-Net, so the figure uses that correct label. ResUNet originally used a 160/40/100 split; TransUNet pooled the official training and test data before an 895/128/256 split; the other pipelines retained the 379-image official test. The runner preserves those source methods. It does not establish an architecture-only comparison. Newly trained peer models are **new runs**, and their picture must not be presented as an output of the old reported checkpoints.

The runner skips the U-Net plotting cell that appeared before its training history was defined, and skips TransUNet's obsolete failed/scaling experiments. It preserves the final architecture, preprocessing, class weighting, epoch limits, and validation-based checkpoint choices in the uploaded source code. All new model files go to `MyDrive/Melanoma_BestCase_Comparison_From_Notebooks`.

Code paths were smoke-checked with synthetic models and the source snippets were checked for syntax. Full GPU training of the three peer pipelines was not run here.


In [ ]:
#@title Run all four pipelines on the selected Attention best-case image
import base64
import gc
import hashlib
import importlib
import json
import sys
import zipfile
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image
import tensorflow as tf
from IPython.display import display
from google.colab import drive, files

drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive')
RUNTIME = Path('/content/melanoma_bestcase_comparison')
RUNTIME.mkdir(parents=True, exist_ok=True)
CASE_DIR = RUNTIME / 'case'
CASE_DIR.mkdir(exist_ok=True)
OUTPUT_ROOT = DRIVE_ROOT / 'Melanoma_BestCase_Comparison_From_Notebooks'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

for gpu in tf.config.list_physical_devices('GPU'):
    try:
        tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError:
        pass

EMBEDDED_FILES = {'ISIC_0000231.jpg': {'base64': '...', 'sha256': '96a88a5e2e30c76de0d4eaea3222fd38792384ec8458d8b00bf8d4503d55572b'}, 'ISIC_0000231_Segmentation.png': {'base64': 'iVBORw0KGgoAAAANSUhEUgAAA8IAAALTCAAAAAD4feT7AAAQNUlEQVR4nO3d2XriyBKFUXG+ev9X5lxU2Y1tbCuFhtwRa11UTzUYKX5lIsC9LAAAAAAAAAAAAAAAAAAAAAAAAAAAAABAK/erv4AWbld/ART1r18DdrT/Xf0FUJMV+CwS5gj3L3/DQSTMAe5P/5YjSJj9yfZEf67+ApjXfd3dqH8/7b+freAzuWHIc+8dfjMi9+X2rNXb14LN2KEcXp566PDJjAyts2bsUJ4LM85OeSISZthYwRbhY0mYJ+6PlVpzpyZhvhJtEAnzxeeC/3ur1f3Zfx783fb++d15osJnXxu6Pf/X660es7c/xFyuZhXmd/flpMXxYb23GK8kYf768YMJrxalx+PYsLB8+HDvQbWtGrSPf/aHt2velmW5m9YnHBTOWiR/n7VPX8eTN2sa2C9spDlrmzv8x9xtwFeQMKd18tsf9OW/K3gF+5L2Tuzkh2lb/VWY2E98Xrix+/L8A4Pn/Mn/3aay2r7CNa2lq5rZ4Za3if3EAekne80zsZ+4ndVOdsHpX/7+JEwYDX8kYdJ4+/QHEu7G/Bcj4WYUXI2Ee6lRcI1HsRMJt2L26/HurD70W5KEuxBwURLuQcBleS7cQrGCvcnygYQ7KFYwjyRMHpekB7YkHVQceZP7jwPRQMWC3785XvsJbn8ASrvu23KcqvcQ9370pdVP913rKXY7q6pGBfcm4aIU3IWEydf6eiXhmloPdS8SLknBfUgYokmYfF5Uohj76EYkXJCCO5EwRJMwRJNwPf320f0e8QMJQzQJQzQJk8/rwlTS+olhQxKGaBKupuEi3HofLWHIJmHiNdx4PJBwMS3HueWDftP7aURBbae57SS3feBFtS14WboOs410Ka0Lbsr/X7gO/bZkFS5DwT2PgFW4hJ7Dy7J0vQNQi37fdRxnG+l4Cu5NwukU/KDjwei486ii47z+qt9A93vENcj3e81mutnDrUG/v+k01p4L51EwDyQcR8E8kjAFdbrMSRiiSThNpwWGFSQM0SQM0SQcxj6ajyScRcHrNDpOEqake5uIJRylzVyymoQhmoSTWIQHdDlYEoZoEg7SZV3ZR5cPHEo4h4J5QsIQTcIxLMJDuuyjJTyzh7cn6JdvtLlWxXmL9qbfTdpMdpsHGka1L+sy2v6fSjMSMKt5LjwhBe+n/rHssttIUn/qznF7O5K1h7z2o4uk4L3VHnIb6dkomCESnoyC91f7mEoYokl4LrUXDA4gYYgmYYgm4anYRzNKwjNRMMMkPBEFH6T0ga39xpUspQftcmUn3SoM0SQ8DYswW0iYHspeISU8i7IjxrEkDNEkPAmL8MHckeZQCmYjCUM0CU/BIny0svto34R2AvrlBXUvTiH0e5Kyk24jfS0F8yIJQzQJX8oifJqyh1rCVyo7VpxHwhdSMK+T8HUUzA4kDNEkfBmL8LmqHm8JQzQJQzQJX6Xqvm5eRY+4hGmjZsMSvkjNcZpcyYNe9vMbUys5ShnqDbxV+AIKvk69Yy9hiCbh89VbCLiQhOml3AVUwhDNt787WblFgItZhc+lYHYmYXop98KwhE9lEb5cuVNQ7po0sXLDE6rYzBd7ODNT8DRKTb2NNEST8FkswhxCwvRT6nIqYYjm3VmnKHXZZypW4TMoeC7uSDNGwRxHwhBNwrRTah8t4RPYR3MgCUM0CdNNrX20hI9nH82RJEwzxRZhCR/OIsyhvMHyUPrlaBI+jn45gY30YRTMGSR8FAVzCgnTTLVrq4QhmoQhmoTppthOWsK0U6thCdPPvVLEEj5IpSFhZt6ddQgBc5ZqH9uYgX4D1Bl8G+ndKZgzSRiiSRiiSRiiSZiO6tzNkjBkkzBEk/B+vJqUo9C5KvSc4Fr/ZuJWajoqKzP5ZR7ItWQbp8zk20jvQcFcRsL0VOayK+EdlJkGAkkYokmYpqrsnST8uiqzQCQJQzQJQzQJQzQJv67M+3xIJGGaqnLllTBNVXkhQcKvqzIL3RQ5bxJ+XZUdWTs1GpYwfZVoWMIQTcI7sJNOVWEZljCdFWhYwhBNwrSWvwxLeAf5Y0AuCUM0Cb/OIpzs/u0/hPB6yMsSTzv/eU/g/umfQ1iFX3G/K7iM1BP55+ovINf9/Qe4jlV4K/GWcn8/n2knVsI0l76bSnvuPo3gc86P0pKwCm+jYCbhdtYG+mUeabuGGSi4sLwgbKSHKbiwvIIlDNkkDO8CF2EJw7vEgiUM2SQM0SQ8LHK3xQqZZ1bCEE3C4zIv1hQlYYgmYfgrdHclYYgm4Q1CL9eUJGFYliX3wizhDXxWqaDUgiW8RezZ5lu551TCEE3CG9hIMw8Jj1MwE5HwMAUzEwlDNAlD8g1pCQ+zj2YqEh4VfL2mIglD9HVZwpBcsITHRZ9vypEw7WVflCVMd9kFSxiySZjuwl/plzDtZTcsYYgm4WHZ12yqkTBEX5UlDNEkPCr6ik09EoZoEoZoEh5kH81cJAzRJAzRJAzRn1WSMESTMESTMO1F76MlPMhrSvVkFyxhyCbhIRZhZiNhiCZhiCZhiCZhmgu/IS1hyCbhEW5IMx0JQzQJQzQJQzQJQzQJQzQJQzQJD/CaUkXpZ1XC66Wfa0qSMESTMN2F764kvFr4maYoCUM0CdNe9v5KwhBNwhBNwmtl77b4Qfb37ZAw7WVfnSW8UvZppi4JQzQJr2MRZlIShmgShmgShmgShug7HRKGaBJeJfoyza+Sz6+EIZqEIZqEIXonLWGIJmGIJmGIJmGIJmGI/sYdEoZoEoZoEoZoEl4j+IV/Vgk+wxKGaBJeI/h+JdVJGKJJGKJJeBU7aWYl4XU0zKSM5nrBLzzwm9wQrMLr5Z5lCpPwAA2XFXxqJQzRJDzAk2HmI2GIJuEBwU+Y+FnwBkvCEE3CEE3CI+yky8rdSUt4yE3ETMZIjsu9YPO92BKswrAswRdmCUM0CcOy2EgD15AwRJPwsNj7Hvwgdh8tYcgmYUhehCU8Lvhs843kcyphSC5YwpBNwrQXvQhLeJwXlZiJhCGahIdlb7uoRsIQTcJ0F76tkjBEkzBEkzBEkzBEkzBEkzBEk/Awb7BkJhKGaBKGaBKGaBKGaBIeFv6WWj4Lvz9pHseFn3KeCO7AKgzLcs+9LksYokkYokkYliX4DoeEIZqExwXfvaQeCcOyBF+YJQzRJLxB7AWbgkzjJrG3L3kuuAOr8CbBZ5wnks+nhCG5YAlDNglDNAlv4nZWJdH7aAlvomCmIeENFMw8JAzRJDzOIsxEJDxMwcxEwqMUzFQkDNEkTHfZLwtLeJR9NHOR8KDwSzblSJjm0i/KEoZoEh6VftHmo/jzGf8ALuCOVhkVxt8qPK7CeWdZipxJCdNWiYIlvEWNU08NEoZoEqarIpspCUO0P1d/AXCFIkvwImHaqRPvX9Uezym8tyPJ7fF8FZx3z4Wp7fbYbcGCJUxtt/cflpoF13xQR7ORjvHffN+rjrpVmB6qFixhyCZhKmvwnKfs9uJQDQajjuojbhWmuOrXWwlDNAlDNAlvUH1rVkzx0yVhiCbhccWv6mSRMESTMEST8DD7aGYiYcqrfdGVMESTMESTMPWV3klLmAYqNyxhiCZhOii8DEt4VOFhIJGEIZqEaaHu5knCg+qOApkkTAt1vwmehCGahCGahCGahOmg7lNhCUM2CUM0CQ8qvCMrrPJZkzBEkzBEk/CoynuyqkqfMwlDNAkPK31JJ4553MTnlZLUHvI/V38BcIBbn8ts7QvUcbrMR6h/Y31fGky4VZi6yue7LG5nbWQRnlun8yNhiCZhiCbhLTrt0zI1OkMShmgShmgShmgS3qDRE61ULV4R/kvCEE3CEE3C4+yjp9doHy3hDTrNR6ZWZ0jClNOqYAlTTq+CfdiQYpoFbBXepN2UBOl3biQM0SQM0SS8Rb/dGtOSMKX0e9+NhCGahDexk2YWEt5Gw7Nqt5OWMEST8EaW4Um1OzES3urWblaYkjl8SbsnXvNrN9HtHvDONDyN23JvOc4dH/OeJDyLtpPsufBr2g7ObPqeiL6PfD9W4gn0HWSr8Ov6Ts88Gp8DCe+g8fxMovMZkDD5OhcsYfK1Lti3vyNd74CtwqTrXrCEyda+YAkTTcESJpmCJUwyBS8S3odR4jIShmgSJpbNz7I4CnvxaaWzmdx/vDuLPPJ9YCNNHAU/kvA+TNV5HOsPJLwTc8U1JLwXDZ/Egf5IwhBNwmSxCH8iYaIo+DMJ78ZwcQUJk8R18gsJQzQJ78cKcTzvRf9CwjvSMOeT8J40fDRH+AsJ78qEcTYJQzTLxs7cbznG7W5Yn3NU9nf/O3DsyJx+y6E5iIb3ZEy/57kw81PwDyR8EFO3H8fyJxKGaBKGaBJmdvbRP5IwRJPwUawdO3EgfyZh5qbgX0gYokkYokn4MHaAe3AUfyNhZqbgX0kYokn4OFYQTiDhA2n4VY7g7yTMvBS8goSPZAQ5nIQPpeFXOHprOEqH8y14NjKbq1iFD2cSOZKEj3cTMccxXSexnR5mNlexCp/EPHIMCUM0CZ/FMjzIAVtHwqcxkhzBXJ3KTa31jOY6jtP5/nV8U/Q3bsvdXK7mUF3ibUQ1/ISZHOJwXUvDXxjJMW5nXcu88iIjdDkL8QcmcpBV+HJm9gNXtEESvp6GeYGEIZqEmYs9ySAJT8DUsp2EIdqfq78AeGBDMswhm4JXUpbFMG7jqM1CxWZxE4dtOm1bNoubuJ01HZPMCPMypfut31psFLdx3GbVrWGTuJGN9KyMNKtIeFq9Gu71aPck4Xn5H7mwgoRnpmF+ZUjm1uSmljHcznukuZp+X2IjPbcG493gIR5KwpMrP+DlH+DRJDy74iNe/OGdQMJcScEvk/D0Kk955cd2FnekuYp+d2EV5iIK3oeE51dz1ms+qgtImEsoeC8S5goK3o2EA9Sb93qP6DruSHOa27Isd/3uzOFMUOPzSmbtEA5rhMCGb/++6L8TdjdoR3FkU4RVbLDO4nZWiqwmsr7aaBKOoQqeMRdBUvbShupMVuEgIWmEfJlVSDjJWxw335+WN0YhzdvLM9Puqo3UuRzvXJNGbKTO5XhHm7FiI3UuxzvcBBG/z5D3P1/BIU93acPG53ruSLOdgifgw4Y8d3u6wH/4twqegbOQbt1G+ja64b49/b3fxsWz3nlYhYv62Oxwb7/8Av3Ow3PhdM9run35D69X5z1hU5JwSbf3Hx7+8cUA9TsnCdfzvlq+v6X601+3/a6v/GKO48Tk+3TT6eMp/fAdb9bf0vr6RmyTMimrcDU/PQVeneHXn6jgWbkjXcvvd5K3vZtLwdNyavK9R7n2ZP5e8ePbnkd+Z87n3OR7K3L9ufyt4Q+/k+8fOzfPhZt70uenV58UPDfnp4Dx3e7jr/i4JBuINM5YDeO73cdf4ZMLwZwylsVtK8g3wff/AAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAAIMX/AWHTmY5QCIzyAAAAAElFTkSuQmCC', 'sha256': 'a0540062d27cfd84a3e711f5c490ba9e6183c35ab476d8bf8dbad4549dae8796'}, 'ISBI2016_ISIC_Part3B_Training_GroundTruth.csv': {'base64': '...', 'sha256': '0ffadd63b7cec650cddfbe5df54d1d3167453fe6004c2bbf05824184b0c4d405'}, 'ISBI2016_ISIC_Part3B_Test_GroundTruth.csv': {'base64': 'SVNJQ18wMDAwMDAzLDAuMA0KSVNJQ18wMDAwMDEyLDAuMA0KSVNJQ18wMDAwMDEzLDEuMA0KSVNJQ18wMDAwMDE0LDAuMA0KSVNJQ18wMDAwMDE1LDAuMA0KSVNJQ18wMDAwMDIwLDAuMA0KSVNJQ18wMDAwMDIyLDEuMA0KSVNJQ18wMDAwMDIzLDAuMA0KSVNJQ18wMDAwMDI3LDAuMA0KSVNJQ18wMDAwMDM2LDEuMA0KSVNJQ18wMDAwMDM3LDAuMA0KSVNJQ18wMDAwMDQwLDEuMA0KSVNJQ18wMDAwMDQzLDEuMA0KSVNJQ18wMDAwMDUyLDAuMA0KSVNJQ18wMDAwMDUzLDAuMA0KSVNJQ18wMDAwMDU2LDEuMA0KSVNJQ18wMDAwMDU3LDAuMA0KSVNJQ18wMDAwMDY0LDAuMA0KSVNJQ18wMDAwMDY2LDAuMA0KSVNJQ18wMDAwMDY5LDAuMA0KSVNJQ18wMDAwMDcxLDAuMA0KSVNJQ18wMDAwMDcyLDAuMA0KSVNJQ18wMDAwMDg4LDAuMA0KSVNJQ18wMDAwMDkyLDAuMA0KSVNJQ18wMDAwMDk4LDAuMA0KSVNJQ18wMDAwMDk5LDAuMA0KSVNJQ18wMDAwMTAxLDAuMA0KSVNJQ18wMDAwMTA3LDAuMA0KSVNJQ18wMDAwMTExLDAuMA0KSVNJQ18wMDAwMTEzLDAuMA0KSVNJQ18wMDAwMTE1LDAuMA0KSVNJQ18wMDAwMTE3LDAuMA0KSVNJQ18wMDAwMTI1LDAuMA0KSVNJQ18wMDAwMTI2LDAuMA0KSVNJQ18wMDAwMTI5LDAuMA0KSVNJQ18wMDAwMTMyLDAuMA0KSVNJQ18wMDAwMTM2LDAuMA0KSVNJQ18wMDAwMTM4LDAuMA0KSVNJQ18wMDAwMTQxLDEuMA0KSVNJQ18wMDAwMTQ5LDEuMA0KSVNJQ18wMDAwMTYwLDEuMA0KSVNJQ18wMDAwMTYxLDEuMA0KSVNJQ18wMDAwMTY0LDEuMA0KSVNJQ18wMDAwMTY1LDEuMA0KSVNJQ18wMDAwMTY5LDEuMA0KSVNJQ18wMDAwMTcyLDEuMA0KSVNJQ18wMDAwMTc0LDEuMA0KSVNJQ18wMDAwMTc3LDAuMA0KSVNJQ18wMDAwMTg4LDAuMA0KSVNJQ18wMDAwMTk1LDAuMA0KSVNJQ18wMDAwMTk2LDAuMA0KSVNJQ18wMDAwMTk3LDAuMA0KSVNJQ18wMDAwMTk4LDAuMA0KSVNJQ18wMDAwMjAwLDAuMA0KSVNJQ18wMDAwMjAxLDAuMA0KSVNJQ18wMDAwMjAyLDAuMA0KSVNJQ18wMDAwMjEyLDAuMA0KSVNJQ18wMDAwMjEzLDAuMA0KSVNJQ18wMDAwMjIyLDAuMA0KSVNJQ18wMDAwMjI2LDAuMA0KSVNJQ18wMDAwMjI3LDAuMA0KSVNJQ18wMDAwMjI4LDAuMA0KSVNJQ18wMDAwMjMwLDAuMA0KSVNJQ18wMDAwMjMxLDAuMA0KSVNJQ18wMDAwMjMzLDAuMA0KSVNJQ18wMDAwMjM0LDAuMA0KSVNJQ18wMDAwMjQ2LDAuMA0KSVNJQ18wMDAwMjUzLDAuMA0KSVNJQ18wMDAwMjU0LDAuMA0KSVNJQ18wMDAwMjU3LDAuMA0KSVNJQ18wMDAwMjU4LDAuMA0KSVNJQ18wMDAwMjcxLDAuMA0KSVNJQ18wMDAwMjc5LDEuMA0KSVNJQ18wMDAwMjg3LDEuMA0KSVNJQ18wMDAwMjg5LDEuMA0KSVNJQ18wMDAwMjk4LDEuMA0KSVNJQ18wMDAwMjk5LDEuMA0KSVNJQ18wMDAwMzAyLDEuMA0KSVNJQ18wMDAwMzA2LDEuMA0KSVNJQ18wMDAwMzEwLDEuMA0KSVNJQ18wMDAwMzExLDEuMA0KSVNJQ18wMDAwMzE5LDAuMA0KSVNJQ18wMDAwMzIwLDAuMA0KSVNJQ18wMDAwMzI1LDAuMA0KSVNJQ18wMDAwMzI3LDAuMA0KSVNJQ18wMDAwMzI4LDAuMA0KSVNJQ18wMDAwMzQwLDAuMA0KSVNJQ18wMDAwMzQzLDAuMA0KSVNJQ18wMDAwMzU0LDAuMA0KSVNJQ18wMDAwMzU3LDAuMA0KSVNJQ18wMDAwMzY4LDAuMA0KSVNJQ18wMDAwMzcxLDAuMA0KSVNJQ18wMDAwMzc3LDAuMA0KSVNJQ18wMDAwMzc4LDAuMA0KSVNJQ18wMDAwMzgwLDAuMA0KSVNJQ18wMDAwMzg4LDAuMA0KSVNJQ18wMDAwMzkyLDAuMA0KSVNJQ18wMDAwMzkzLDEuMA0KSVNJQ18wMDAwNDA3LDAuMA0KSVNJQ18wMDAwNDE4LDAuMA0KSVNJQ18wMDAwNDIwLDEuMA0KSVNJQ18wMDAwNDMzLDAuMA0KSVNJQ18wMDAwNDQ4LDAuMA0KSVNJQ18wMDAwNDQ5LDAuMA0KSVNJQ18wMDAwNDU2LDAuMA0KSVNJQ18wMDAwNDY2LDEuMA0KSVNJQ18wMDAwNDcwLDAuMA0KSVNJQ18wMDAwNDc2LDAuMA0KSVNJQ18wMDAwNDc5LDAuMA0KSVNJQ18wMDAwNDgyLDEuMA0KSVNJQ18wMDAwNDg0LDEuMA0KSVNJQ18wMDAwNDg3LDEuMA0KSVNJQ18wMDAwNDkwLDAuMA0KSVNJQ18wMDAwNDk0LDAuMA0KSVNJQ18wMDAwNDk5LDAuMA0KSVNJQ18wMDAwNTAxLDAuMA0KSVNJQ18wMDAwNTA4LDAuMA0KSVNJQ18wMDAwNTA5LDAuMA0KSVNJQ18wMDAwNTEwLDAuMA0KSVNJQ18wMDAwNTE1LDAuMA0KSVNJQ18wMDAwNTE4LDEuMA0KSVNJQ18wMDAwNTI1LDAuMA0KSVNJQ18wMDAwNTI3LDAuMA0KSVNJQ18wMDAwNTMzLDEuMA0KSVNJQ18wMDAwNTM0LDAuMA0KSVNJQ18wMDAwNTM3LDAuMA0KSVNJQ18wMDAwNTM5LDAuMA0KSVNJQ18wMDAwNTQwLDAuMA0KSVNJQ18wMDAwNTQ3LDEuMA0KSVNJQ18wMDAwNTQ5LDEuMA0KSVNJQ18wMDAwNTUwLDEuMA0KSVNJQ18wMDAxMTAwLDEuMA0KSVNJQ18wMDAxMTAzLDEuMA0KSVNJQ18wMDAxMTI4LDEuMA0KSVNJQ18wMDAxMTMxLDEuMA0KSVNJQ18wMDAxMTQyLDEuMA0KSVNJQ18wMDAxMTgxLDAuMA0KSVNJQ18wMDAxMTg1LDAuMA0KSVNJQ18wMDAxMTg2LDAuMA0KSVNJQ18wMDAxMTkwLDAuMA0KSVNJQ18wMDAxMjA0LDAuMA0KSVNJQ18wMDAxMjQyLDAuMA0KSVNJQ18wMDAxMjk5LDAuMA0KSVNJQ18wMDAxNDI3LDAuMA0KSVNJQ18wMDAxNDg0LDAuMA0KSVNJQ18wMDAxNjg1LDAuMA0KSVNJQ18wMDAxOTYwLDAuMA0KSVNJQ18wMDAyMTA3LDAuMA0KSVNJQ18wMDAyMjQ2LDAuMA0KSVNJQ18wMDAyNjczLDAuMA0KSVNJQ18wMDAyODI5LDAuMA0KSVNJQ18wMDAyODcxLDAuMA0KSVNJQ18wMDAzMDU2LDAuMA0KSVNJQ18wMDAzNTU5LDAuMA0KSVNJQ18wMDAzNzI4LDAuMA0KSVNJQ18wMDA0MTE1LDAuMA0KSVNJQ18wMDA0MzQ2LDAuMA0KSVNJQ18wMDA1MDAwLDAuMA0KSVNJQ18wMDA2MTkzLDAuMA0KSVNJQ18wMDA2NzExLDAuMA0KSVNJQ18wMDA3MzIyLDAuMA0KSVNJQ18wMDA3NjkzLDAuMA0KSVNJQ18wMDA4MTE2LDAuMA0KSVNJQ18wMDA4MjA3LDAuMA0KSVNJQ18wMDA4NDA2LDAuMA0KSVNJQ18wMDA4NTA3LDAuMA0KSVNJQ18wMDA4NjAwLDAuMA0KSVNJQ18wMDA4NjI2LDAuMA0KSVNJQ18wMDA4NjU5LDAuMA0KSVNJQ18wMDA4OTkyLDAuMA0KSVNJQ18wMDA4OTk4LDAuMA0KSVNJQ18wMDA5MDM1LDAuMA0KSVNJQ18wMDA5MDc4LDAuMA0KSVNJQ18wMDA5MDgzLDAuMA0KSVNJQ18wMDA5MjAxLDAuMA0KSVNJQ18wMDA5Mjk4LDAuMA0KSVNJQ18wMDA5NTY0LDAuMA0KSVNJQ18wMDA5ODY5LDAuMA0KSVNJQ18wMDA5ODcyLDAuMA0KSVNJQ18wMDA5ODc0LDAuMA0KSVNJQ18wMDA5ODc5LDAuMA0KSVNJQ18wMDA5ODgwLDAuMA0KSVNJQ18wMDA5ODgxLDAuMA0KSVNJQ18wMDA5ODgyLDEuMA0KSVNJQ18wMDA5ODg1LDAuMA0KSVNJQ18wMDA5ODg5LDAuMA0KSVNJQ18wMDA5ODkxLDAuMA0KSVNJQ18wMDA5ODk4LDEuMA0KSVNJQ18wMDA5OTAxLDAuMA0KSVNJQ18wMDA5OTAyLDAuMA0KSVNJQ18wMDA5OTA2LDAuMA0KSVNJQ18wMDA5OTE4LDEuMA0KSVNJQ18wMDA5OTIwLDAuMA0KSVNJQ18wMDA5OTIzLDAuMA0KSVNJQ18wMDA5OTI2LDAuMA0KSVNJQ18wMDA5OTI3LDEuMA0KSVNJQ18wMDA5OTI4LDAuMA0KSVNJQ18wMDA5OTMwLDAuMA0KSVNJQ18wMDA5OTMxLDAuMA0KSVNJQ18wMDA5OTQzLDAuMA0KSVNJQ18wMDA5OTQ1LDAuMA0KSVNJQ18wMDA5OTQ2LDAuMA0KSVNJQ18wMDA5OTQ4LDAuMA0KSVNJQ18wMDA5OTU0LDAuMA0KSVNJQ18wMDA5OTU1LDEuMA0KSVNJQ18wMDA5OTU2LDAuMA0KSVNJQ18wMDA5OTU4LDAuMA0KSVNJQ18wMDA5OTU5LDEuMA0KSVNJQ18wMDA5OTY1LDAuMA0KSVNJQ18wMDA5OTcwLDAuMA0KSVNJQ18wMDA5OTc3LDAuMA0KSVNJQ18wMDA5OTgwLDAuMA0KSVNJQ18wMDA5OTgyLDAuMA0KSVNJQ18wMDA5OTg4LDAuMA0KSVNJQ18wMDA5OTkwLDAuMA0KSVNJQ18wMDA5OTkyLDEuMA0KSVNJQ18wMDA5OTkzLDEuMA0KSVNJQ18wMDA5OTk0LDAuMA0KSVNJQ18wMDA5OTk4LDAuMA0KSVNJQ18wMDEwMDA0LDAuMA0KSVNJQ18wMDEwMDA5LDAuMA0KSVNJQ18wMDEwMDExLDEuMA0KSVNJQ18wMDEwMDEzLDAuMA0KSVNJQ18wMDEwMDE2LDEuMA0KSVNJQ18wMDEwMDE4LDAuMA0KSVNJQ18wMDEwMDIwLDAuMA0KSVNJQ18wMDEwMDIzLDEuMA0KSVNJQ18wMDEwMDI4LDAuMA0KSVNJQ18wMDEwMDMzLDAuMA0KSVNJQ18wMDEwMDM0LDEuMA0KSVNJQ18wMDEwMDM3LDAuMA0KSVNJQ18wMDEwMDM4LDAuMA0KSVNJQ18wMDEwMDQxLDAuMA0KSVNJQ18wMDEwMDQ3LDAuMA0KSVNJQ18wMDEwMDU1LDAuMA0KSVNJQ18wMDEwMDU4LDAuMA0KSVNJQ18wMDEwMDU5LDAuMA0KSVNJQ18wMDEwMDYxLDAuMA0KSVNJQ18wMDEwMDYyLDAuMA0KSVNJQ18wMDEwMDczLDAuMA0KSVNJQ18wMDEwMDc3LDAuMA0KSVNJQ18wMDEwMDg4LDAuMA0KSVNJQ18wMDEwMDg5LDEuMA0KSVNJQ18wMDEwMDkyLDAuMA0KSVNJQ18wMDEwMTAwLDAuMA0KSVNJQ18wMDEwMTAzLDAuMA0KSVNJQ18wMDEwMTcxLDEuMA0KSVNJQ18wMDEwMTczLDAuMA0KSVNJQ18wMDEwMTc1LDEuMA0KSVNJQ18wMDEwMTgwLDAuMA0KSVNJQ18wMDEwMTgzLDAuMA0KSVNJQ18wMDEwMTg3LDAuMA0KSVNJQ18wMDEwMTkwLDAuMA0KSVNJQ18wMDEwMTkyLDAuMA0KSVNJQ18wMDEwMTkzLDAuMA0KSVNJQ18wMDEwMjAxLDAuMA0KSVNJQ18wMDEwMjAyLDAuMA0KSVNJQ18wMDEwMjA2LDAuMA0KSVNJQ18wMDEwMjA3LDEuMA0KSVNJQ18wMDEwMjE1LDAuMA0KSVNJQ18wMDEwMjE2LDAuMA0KSVNJQ18wMDEwMjI5LDAuMA0KSVNJQ18wMDEwMjMxLDEuMA0KSVNJQ18wMDEwMjM0LDAuMA0KSVNJQ18wMDEwMjM4LDAuMA0KSVNJQ18wMDEwMjU0LDAuMA0KSVNJQ18wMDEwMjU1LDAuMA0KSVNJQ18wMDEwMjU3LDAuMA0KSVNJQ18wMDEwMjYxLDAuMA0KSVNJQ18wMDEwMzI2LDAuMA0KSVNJQ18wMDEwMzMxLDAuMA0KSVNJQ18wMDEwMzM2LDAuMA0KSVNJQ18wMDEwMzQwLDAuMA0KSVNJQ18wMDEwMzQ2LDAuMA0KSVNJQ18wMDEwMzQ3LDAuMA0KSVNJQ18wMDEwMzQ4LDAuMA0KSVNJQ18wMDEwMzYwLDEuMA0KSVNJQ18wMDEwMzY4LDAuMA0KSVNJQ18wMDEwMzY5LDEuMA0KSVNJQ18wMDEwMzczLDAuMA0KSVNJQ18wMDEwMzc0LDAuMA0KSVNJQ18wMDEwMzc3LDAuMA0KSVNJQ18wMDEwMzc4LDAuMA0KSVNJQ18wMDEwMzc5LDEuMA0KSVNJQ18wMDEwNDM3LDAuMA0KSVNJQ18wMDEwNDQ0LDAuMA0KSVNJQ18wMDEwNDQ4LDAuMA0KSVNJQ18wMDEwNDQ5LDAuMA0KSVNJQ18wMDEwNDUxLDAuMA0KSVNJQ18wMDEwNDUyLDAuMA0KSVNJQ18wMDEwNDU0LDEuMA0KSVNJQ18wMDEwNDU2LDAuMA0KSVNJQ18wMDEwNDYwLDAuMA0KSVNJQ18wMDEwNDYzLDAuMA0KSVNJQ18wMDEwNDc0LDAuMA0KSVNJQ18wMDEwNDc3LDAuMA0KSVNJQ18wMDEwNDgzLDAuMA0KSVNJQ18wMDEwNDk0LDAuMA0KSVNJQ18wMDEwNDk4LDAuMA0KSVNJQ18wMDEwNTUyLDAuMA0KSVNJQ18wMDEwNTUzLDAuMA0KSVNJQ18wMDEwNTU2LDAuMA0KSVNJQ18wMDEwNTY1LDAuMA0KSVNJQ18wMDEwNTc0LDAuMA0KSVNJQ18wMDEwNTgyLDAuMA0KSVNJQ18wMDEwNTg0LDAuMA0KSVNJQ18wMDEwNTg3LDEuMA0KSVNJQ18wMDEwNTg4LDAuMA0KSVNJQ18wMDEwNTkxLDEuMA0KSVNJQ18wMDEwNTkyLDAuMA0KSVNJQ18wMDEwNTk2LDEuMA0KSVNJQ18wMDEwNTk3LDEuMA0KSVNJQ18wMDEwNTk4LDAuMA0KSVNJQ18wMDEwNTk5LDAuMA0KSVNJQ18wMDEwNjA0LDAuMA0KSVNJQ18wMDEwNjA2LDAuMA0KSVNJQ18wMDEwNjA3LDAuMA0KSVNJQ18wMDEwODQ2LDAuMA0KSVNJQ18wMDEwODQ3LDAuMA0KSVNJQ18wMDEwODU0LDAuMA0KSVNJQ18wMDEwODU2LDAuMA0KSVNJQ18wMDEwODYzLDEuMA0KSVNJQ18wMDExMDgwLDAuMA0KSVNJQ18wMDExMDgxLDAuMA0KSVNJQ18wMDExMDgzLDAuMA0KSVNJQ18wMDExMDg5LDAuMA0KSVNJQ18wMDExMDkwLDAuMA0KSVNJQ18wMDExMDkyLDAuMA0KSVNJQ18wMDExMDk4LDAuMA0KSVNJQ18wMDExMTAxLDAuMA0KSVNJQ18wMDExMTA0LDAuMA0KSVNJQ18wMDExMTA3LDAuMA0KSVNJQ18wMDExMTEwLDAuMA0KSVNJQ18wMDExMTEyLDEuMA0KSVNJQ18wMDExMTIyLDEuMA0KSVNJQ18wMDExMTI5LDAuMA0KSVNJQ18wMDExMTMyLDEuMA0KSVNJQ18wMDExMTQzLDAuMA0KSVNJQ18wMDExMTQ4LDAuMA0KSVNJQ18wMDExMTQ5LDAuMA0KSVNJQ18wMDExMTUwLDAuMA0KSVNJQ18wMDExMTUxLDEuMA0KSVNJQ18wMDExMTUyLDAuMA0KSVNJQ18wMDExMTU1LDAuMA0KSVNJQ18wMDExMTYyLDAuMA0KSVNJQ18wMDExMTY3LDAuMA0KSVNJQ18wMDExMTY4LDAuMA0KSVNJQ18wMDExMTcxLDAuMA0KSVNJQ18wMDExMTcyLDAuMA0KSVNJQ18wMDExMTc1LDAuMA0KSVNJQ18wMDExMTc2LDEuMA0KSVNJQ18wMDExMTc3LDAuMA0KSVNJQ18wMDExMjA0LDAuMA0KSVNJQ18wMDExMjA1LDAuMA0KSVNJQ18wMDExMjEzLDAuMA0KSVNJQ18wMDExMjE5LDAuMA0KSVNJQ18wMDExMjI0LDAuMA0KSVNJQ18wMDExMjI3LDAuMA0KSVNJQ18wMDExMjkyLDAuMA0KSVNJQ18wMDExMjk0LDAuMA0KSVNJQ18wMDExMjk4LDAuMA0KSVNJQ18wMDExMzAwLDAuMA0KSVNJQ18wMDExMzA1LDEuMA0KSVNJQ18wMDExMzEwLDAuMA0KSVNJQ18wMDExMzE5LDAuMA0KSVNJQ18wMDExMzIxLDAuMA0KSVNJQ18wMDExMzI1LDAuMA0KSVNJQ18wMDExMzMzLDAuMA0KSVNJQ18wMDExMzM2LDAuMA0KSVNJQ18wMDExMzM4LDAuMA0KSVNJQ18wMDExMzQ0LDEuMA0KSVNJQ18wMDExMzQ5LDEuMA0KSVNJQ18wMDExMzU5LDAuMA0KSVNJQ18wMDExMzYzLDAuMA0KSVNJQ18wMDExMzY3LDAuMA0KSVNJQ18wMDExMzc0LDEuMA0KSVNJQ18wMDExMzg0LDAuMA0KSVNJQ18wMDExMzg2LDAuMA0KSVNJQ18wMDExMzkyLDAuMA0K', 'sha256': '8f7718bfe5ed110e0235b0d9afc40fe89e52610c5687cc9cd7dd0c23352b82ce'}}

for name, item in EMBEDDED_FILES.items():
    data = base64.b64decode(item['base64'])
    if hashlib.sha256(data).hexdigest() != item['sha256']:
        raise ValueError('Embedded file checksum failed: ' + name)
    (CASE_DIR / name).write_bytes(data)
IMAGE_ID = 'ISIC_0000231'
IMAGE_PATH = CASE_DIR / f'{IMAGE_ID}.jpg'
MASK_PATH = CASE_DIR / f'{IMAGE_ID}_Segmentation.png'
labels = pd.read_csv(CASE_DIR / 'ISBI2016_ISIC_Part3B_Test_GroundTruth.csv',
                     header=None, names=['image_id', 'label'])
assert int(labels.loc[labels.image_id == IMAGE_ID, 'label'].iloc[0]) == 0

MODULE_DIR = RUNTIME / 'modules'
MODULE_DIR.mkdir(exist_ok=True)
MODULES = {'attention_pipeline_core.py': '"""Inference and Gradio UI for the frozen Attention U-Net melanoma candidate.\n\nNo training or threshold selection occurs in this module. JPEG ROI encoding is\npreserved because the training notebook classified saved quality-95 JPEG ROIs.\n"""\nfrom __future__ import annotations\n\nfrom io import BytesIO\nfrom pathlib import Path\nimport hashlib\nimport json\nimport threading\n\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nfrom scipy import ndimage\nimport tensorflow as tf\n\nSEG_SIZE = (256, 256)\nCLS_SIZE = (224, 224)\n\n\ndef image_tensor(path, size, normalise=False):\n    pixels = tf.io.decode_image(tf.io.read_file(str(path)), channels=3,\n                                expand_animations=False)\n    pixels = tf.cast(tf.image.resize(pixels, size, method="bilinear"), tf.float32)\n    return pixels / 255.0 if normalise else pixels\n\n\ndef roi_tensor(roi):\n    # Match roi.save(..., quality=95) -> decode_classifier_image() in training.\n    buffer = BytesIO()\n    roi.convert("RGB").save(buffer, format="JPEG", quality=95)\n    pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n    return tf.cast(tf.image.resize(pixels, CLS_SIZE), tf.float32)\n\n\ndef crop_from_prediction(image_pil, predicted_mask, padding=0.15):\n    image = image_pil.convert("RGB")\n    width, height = image.size\n    binary = Image.fromarray(np.uint8(np.asarray(predicted_mask) > 0) * 255)\n    binary = np.asarray(binary.resize(image.size, Image.Resampling.NEAREST)) > 0\n    labelled, count = ndimage.label(binary)\n    reason = "cropped"\n    if count == 0:\n        crop, reason = image, "empty predicted mask"\n    else:\n        sizes = np.bincount(labelled.ravel())\n        sizes[0] = 0\n        lesion = labelled == int(sizes.argmax())\n        area = int(lesion.sum())\n        if area < max(64, int(width * height * 0.001)) or area > width * height * 0.80:\n            crop, reason = image, "implausible predicted area"\n        else:\n            ys, xs = np.where(lesion)\n            span = max(np.ptp(xs) + 1, np.ptp(ys) + 1)\n            margin = int(np.ceil(span * padding))\n            x0 = max(0, int(xs.min()) - margin)\n            x1 = min(width, int(xs.max()) + margin + 1)\n            y0 = max(0, int(ys.min()) - margin)\n            y1 = min(height, int(ys.max()) + margin + 1)\n            crop = image.crop((x0, y0, x1, y1))\n    a = np.asarray(crop)\n    h, w = a.shape[:2]\n    side = max(h, w)\n    top, left = (side - h) // 2, (side - w) // 2\n    square = np.pad(a, ((top, side - h - top), (left, side - w - left), (0, 0)),\n                    mode="edge")\n    roi = Image.fromarray(square).resize((CLS_SIZE[1], CLS_SIZE[0]),\n                                         Image.Resampling.BILINEAR)\n    return roi, reason\n\n\ndef mask_overlay(image, mask):\n    base = np.asarray(image.convert("RGB"), dtype=np.float32)\n    mask = Image.fromarray(np.uint8(mask) * 255).resize(\n        image.size, Image.Resampling.NEAREST)\n    binary = np.asarray(mask) > 127\n    tint = np.array([35, 195, 145], dtype=np.float32)\n    base[binary] = 0.65 * base[binary] + 0.35 * tint\n    boundary = binary & ~ndimage.binary_erosion(binary)\n    base[boundary] = [255, 215, 30]\n    return Image.fromarray(np.uint8(np.clip(base, 0, 255)))\n\n\ndef gradcam_overlay(roi, heatmap):\n    from matplotlib import colormaps\n    image = roi.convert("RGB").resize((CLS_SIZE[1], CLS_SIZE[0]))\n    heat = Image.fromarray(np.uint8(np.clip(heatmap, 0, 1) * 255)).resize(\n        image.size, Image.Resampling.BILINEAR)\n    colors = colormaps["jet"](np.asarray(heat, dtype=np.float32) / 255.0)[..., :3]\n    base = np.asarray(image, dtype=np.float32) / 255.0\n    return Image.fromarray(np.uint8(255 * np.clip(0.62 * base + 0.38 * colors, 0, 1)))\n\n\nclass MelanomaPipeline:\n    def __init__(self, segmenter, classifier, configuration, run_dir=None):\n        self.segmenter = segmenter\n        self.classifier = classifier\n        self.configuration = dict(configuration)\n        self.run_dir = Path(run_dir) if run_dir else None\n        self.lock = threading.RLock()\n        cfg = self.configuration\n        if cfg.get("view") != "dual":\n            raise ValueError("This app requires the full-image + ROI classifier (view=\'dual\').")\n        for field in ("threshold", "segmentation_threshold"):\n            if field not in cfg or not 0 <= float(cfg[field]) <= 1:\n                raise ValueError(f"Missing or invalid saved configuration field: {field}")\n        for field in ("method", "segmentation_method"):\n            if cfg.get(field) not in ("single", "4-flip average"):\n                raise ValueError(f"Unsupported inference method: {cfg.get(field)}")\n        if segmenter.input_shape[1:] != (*SEG_SIZE, 3):\n            raise ValueError("Segmenter input must be 256 x 256 RGB.")\n        if len(classifier.inputs) != 2:\n            raise ValueError("Expected two classifier inputs: full image and ROI.")\n        shapes = [tuple(x.shape[1:]) for x in classifier.inputs]\n        if shapes != [(*CLS_SIZE, 3), (*CLS_SIZE, 3)]:\n            raise ValueError(f"Classifier input shapes are incompatible: {shapes}")\n\n    @classmethod\n    def load(cls, run_dir):\n        run_dir = Path(run_dir)\n        paths = {\n            "configuration": run_dir / "tables" / "selected_configuration.json",\n            "segmenter": run_dir / "models" / "attention_unet_best.keras",\n            "classifier": run_dir / "models" / "selected_final_classifier.keras",\n        }\n        missing = [str(p) for p in paths.values() if not p.is_file()]\n        if missing:\n            raise FileNotFoundError("The saved pipeline is incomplete. Missing:\\n" + "\\n".join(missing))\n        configuration = json.loads(paths["configuration"].read_text())\n        provenance = run_dir / "tables" / "segmentation_configuration.json"\n        if provenance.is_file():\n            expected = json.loads(provenance.read_text()).get("checkpoint_sha256")\n            if expected:\n                actual = hashlib.sha256(paths["segmenter"].read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError("The segmentation checkpoint does not match its saved configuration.")\n        manifest_path = run_dir / "deployment_manifest.json"\n        if manifest_path.is_file():\n            for relative, expected in json.loads(manifest_path.read_text()).get("sha256", {}).items():\n                actual = hashlib.sha256((run_dir / relative).read_bytes()).hexdigest()\n                if actual != expected:\n                    raise ValueError(f"Deployment bundle file has changed: {relative}")\n        # compile=False avoids needing custom training losses for inference.\n        segmenter = tf.keras.models.load_model(paths["segmenter"], compile=False)\n        classifier = tf.keras.models.load_model(paths["classifier"], compile=False)\n        return cls(segmenter, classifier, configuration, run_dir)\n\n    def segmentation_probabilities(self, images, batch_size=4):\n        images = np.asarray(images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "segmentation_method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            transformed = np.flip(images, axis=axes) if axes else images\n            score = self.segmenter.predict(np.ascontiguousarray(transformed),\n                                            batch_size=batch_size, verbose=0)[..., 0]\n            score = np.flip(score, axis=axes) if axes else score\n            predictions.append(score)\n        return np.mean(np.stack(predictions), axis=0)\n\n    def classifier_probabilities(self, full_images, roi_images, batch_size=8):\n        full_images = np.asarray(full_images, dtype=np.float32)\n        roi_images = np.asarray(roi_images, dtype=np.float32)\n        methods = [(), (2,), (1,), (1, 2)] if self.configuration[\n            "method"] == "4-flip average" else [()]\n        predictions = []\n        for axes in methods:\n            full = np.flip(full_images, axis=axes) if axes else full_images\n            roi = np.flip(roi_images, axis=axes) if axes else roi_images\n            features = {"full_image": np.ascontiguousarray(full),\n                        "roi_image": np.ascontiguousarray(roi)}\n            predictions.append(self.classifier.predict(\n                features, batch_size=batch_size, verbose=0).ravel())\n        return np.mean(np.stack(predictions), axis=0)\n\n    def gradcam(self, full_batch, roi_batch, target_class):\n        model = self.classifier\n        backbone = model.get_layer("efficientnetb0")\n        gap = model.get_layer("gap")\n        with tf.GradientTape() as tape:\n            full_map = backbone(full_batch, training=False)\n            roi_map = backbone(roi_batch, training=False)\n            tape.watch(roi_map)\n            features = model.get_layer("combine_features")([gap(full_map), gap(roi_map)])\n            hidden = model.get_layer("hidden_dense")(features)\n            hidden = model.get_layer("head_dropout")(hidden, training=False)\n            output = model.get_layer("prediction")\n            logit = tf.linalg.matmul(hidden, output.kernel) + output.bias\n            score = logit[:, 0] if target_class == 1 else -logit[:, 0]\n        gradients = tape.gradient(score, roi_map)\n        if gradients is None:\n            raise RuntimeError("Grad-CAM could not trace the ROI feature map.")\n        weights = tf.reduce_mean(gradients, axis=(1, 2), keepdims=True)\n        heatmap = tf.nn.relu(tf.reduce_sum(roi_map * weights, axis=-1)[0])\n        return (heatmap / (tf.reduce_max(heatmap) + 1e-8)).numpy()\n\n    def predict_file(self, path, explain=True):\n        if path is None:\n            raise ValueError("Upload a dermoscopic lesion image first.")\n        path = Path(path)\n        with Image.open(path) as handle:\n            original = handle.convert("RGB")\n        with self.lock:\n            segmentation_input = image_tensor(path, SEG_SIZE, normalise=True)[None, ...]\n            probabilities = self.segmentation_probabilities(segmentation_input)[0]\n            mask = probabilities >= float(self.configuration["segmentation_threshold"])\n            roi, reason = crop_from_prediction(\n                original, mask, padding=float(self.configuration.get("roi_padding", 0.15)))\n            full_input = image_tensor(path, CLS_SIZE)[None, ...]\n            roi_input = roi_tensor(roi)[None, ...]\n            score = float(self.classifier_probabilities(full_input, roi_input)[0])\n            predicted = int(score >= float(self.configuration["threshold"]))\n            heatmap = self.gradcam(full_input, roi_input, predicted) if explain else None\n        result = {\n            "original": original, "mask": Image.fromarray(np.uint8(mask) * 255),\n            "mask_overlay": mask_overlay(original, mask), "roi": roi,\n            "score": score, "predicted_label": predicted,\n            "label": "Melanoma" if predicted else "Benign",\n            "roi_reason": reason, "probability_map": probabilities,\n        }\n        result["gradcam"] = gradcam_overlay(roi, heatmap) if heatmap is not None else None\n        return result\n\n\ndef build_demo(pipeline, sample_paths=()):\n    import gradio as gr\n    cfg = pipeline.configuration\n\n    def preview(path):\n        if path is None:\n            return None\n        with Image.open(path) as image:\n            return image.convert("RGB")\n\n    def analyse(path):\n        try:\n            result = pipeline.predict_file(path)\n        except (ValueError, FileNotFoundError, OSError) as error:\n            raise gr.Error(str(error)) from error\n        summary = f"### Model prediction: **{result[\'label\']}**\\n"\n        summary += (f"Melanoma model score: **{result[\'score\']:.4f}**. "\n                    f"Decision threshold: {float(cfg[\'threshold\']):.4f}.\\n\\n"\n                    "The score is a model output, not a calibrated clinical probability.")\n        if result["roi_reason"] != "cropped":\n            summary += "\\n\\nThe lesion crop was unreliable; the classifier used the full image for the ROI view."\n        panels = [(result["original"], "Original image"),\n                  (result["mask"], "Predicted lesion mask"),\n                  (result["mask_overlay"], "Lesion outline"),\n                  (result["roi"], "Lesion ROI"),\n                  (result["gradcam"], "Grad-CAM on ROI")]\n        return summary, panels\n\n    def clear():\n        return None, None, "Upload a dermoscopic lesion image, then select Analyse.", []\n\n    with gr.Blocks(title="Attention U-Net melanoma research demo",\n                   delete_cache=(3600, 3600)) as demo:\n        gr.Markdown("# Melanoma image analysis\\n"\n                    "Attention U-Net lesion segmentation with EfficientNetB0 classification.\\n\\n"\n                    "**Student research prototype. Not validated for clinical diagnosis.**")\n        with gr.Tab("Analyse image"):\n            with gr.Row():\n                with gr.Column(scale=1):\n                    upload = gr.File(label="Upload dermoscopic image (JPG or PNG)",\n                                     file_types=[".jpg", ".jpeg", ".png"], type="filepath")\n                    input_preview = gr.Image(label="Uploaded image", interactive=False)\n                    with gr.Row():\n                        analyse_button = gr.Button("Analyse", variant="primary")\n                        clear_button = gr.Button("Clear")\n                    if sample_paths:\n                        gr.Examples(examples=[[str(p)] for p in sample_paths],\n                                    inputs=upload, label="Public ISIC example images")\n                with gr.Column(scale=2):\n                    summary = gr.Markdown("Upload a dermoscopic lesion image, then select Analyse.")\n                    panels = gr.Gallery(label="Pipeline outputs", columns=3,\n                                        object_fit="contain", height="auto")\n                    gr.Markdown("The lesion mask estimates the lesion boundary. "\n                                "Grad-CAM shows image regions contributing to the model prediction; "\n                                "it is not a lesion mask.")\n            upload.change(preview, inputs=upload, outputs=input_preview)\n            analyse_button.click(analyse, inputs=upload, outputs=[summary, panels],\n                                 api_name="analyse", concurrency_limit=1)\n            clear_button.click(clear, outputs=[upload, input_preview, summary, panels])\n        with gr.Tab("Evaluation"):\n            gr.Markdown("## Saved evaluation results\\n"\n                        "Classification and segmentation measure different tasks. "\n                        "The official ISIC test set was viewed in earlier experiments; "\n                        "these results are exploratory follow-up.")\n            run_dir = pipeline.run_dir\n            displayed = False\n            if run_dir:\n                for filename, label in [("complete_segmentation_test_metrics.csv", "Lesion segmentation"),\n                                        ("complete_classification_test_metrics.csv", "Melanoma classification")]:\n                    path = run_dir / "tables" / filename\n                    if path.is_file():\n                        gr.Markdown("### " + label)\n                        gr.Dataframe(value=pd.read_csv(path).T.reset_index(),\n                                     headers=["Metric", "Value"], interactive=False)\n                        displayed = True\n                figures = sorted((run_dir / "figures").glob("*.png"))\n                if figures:\n                    gr.Gallery(value=[(str(p), p.stem.replace("_", " ")) for p in figures],\n                               label="Evaluation figures", columns=1, object_fit="contain")\n                    displayed = True\n            if not displayed:\n                gr.Markdown("Run the reporting notebook to add test loss, metrics and evaluation figures.")\n    return demo\n', 'bestcase_comparison_helpers.py': '\nfrom pathlib import Path\nfrom io import BytesIO\nimport gc\nimport json\nimport numpy as np\nimport pandas as pd\nfrom PIL import Image\nimport tensorflow as tf\nimport cv2\nfrom attention_pipeline_core import MelanomaPipeline\n\nIMG_SIZE = 256\nCLS_IMG_SIZE = 224\nMASK_MODE = "mask_crop"\nSEG_IMAGE_SIZE = (256, 256)\nSEG_THRESHOLD = 0.50\n\n\ndef get_lesion_region(image, unet_model, pad_frac=0.15, mode=MASK_MODE):\n    """Runs the trained U-Net on a full-res image to get a lesion mask, then applies\n    pixel-masking and/or cropping (per `mode`) before resizing for the classifier."""\n    h, w = image.shape[0], image.shape[1]\n    seg_input = tf.image.resize(image, [IMG_SIZE, IMG_SIZE]) / 255.0\n    seg_input = tf.expand_dims(seg_input, 0)\n    pred_mask_small = unet_model(seg_input, training=False)[0, ..., 0].numpy()\n    pred_mask_small = (pred_mask_small > 0.5).astype(np.uint8)\n\n    # Upsample the mask back to the original image resolution for pixel-accurate masking\n    pred_mask_full = tf.image.resize(\n        pred_mask_small[..., np.newaxis], [h, w], method="nearest"\n    ).numpy()[..., 0].astype(np.uint8)\n\n    ys, xs = np.where(pred_mask_small > 0)\n    lesion_found = len(xs) > 0 and len(ys) > 0\n\n    work_image = image.numpy() if hasattr(image, "numpy") else np.array(image)\n\n    if mode in ("mask", "mask_crop"):\n        if lesion_found:\n            work_image = work_image * pred_mask_full[..., np.newaxis]\n        # if no lesion detected, fall back to the unmasked image rather than an all-black frame\n\n    if mode in ("crop", "mask_crop") and lesion_found:\n        x_min, x_max = xs.min() / IMG_SIZE, xs.max() / IMG_SIZE\n        y_min, y_max = ys.min() / IMG_SIZE, ys.max() / IMG_SIZE\n        pad_x = (x_max - x_min) * pad_frac\n        pad_y = (y_max - y_min) * pad_frac\n        x_min = max(0.0, x_min - pad_x); x_max = min(1.0, x_max + pad_x)\n        y_min = max(0.0, y_min - pad_y); y_max = min(1.0, y_max + pad_y)\n\n        x1, x2 = int(x_min * w), int(x_max * w)\n        y1, y2 = int(y_min * h), int(y_max * h)\n        work_image = work_image[y1:max(y2, y1 + 1), x1:max(x2, x1 + 1), :]\n\n    out = tf.image.resize(work_image, [CLS_IMG_SIZE, CLS_IMG_SIZE])\n    return out.numpy()\n\ndef predict_mask_for_pil(model, image_pil):\n    resized = image_pil.resize(\n        (SEG_IMAGE_SIZE[1], SEG_IMAGE_SIZE[0]), Image.Resampling.BILINEAR\n    )\n    x = np.asarray(resized, dtype=np.float32) / 255.0\n    probability = model.predict(x[None, ...], verbose=0)[0, ..., 0]\n    binary = (probability >= SEG_THRESHOLD).astype(np.uint8)\n    return probability, binary\n\ndef extract_roi_from_binary_mask(image_pil, binary_mask, padding_fraction=0.10):\n    image_array = np.asarray(image_pil.convert(\'RGB\'))\n    h, w = image_array.shape[:2]\n    mask = np.asarray(\n        Image.fromarray((binary_mask * 255).astype(np.uint8)).resize(\n            (w, h), Image.Resampling.NEAREST\n        )\n    ) > 0\n\n    ys, xs = np.where(mask)\n    if len(xs) == 0 or len(ys) == 0:\n        return image_pil.copy(), (0, 0, w, h), True\n\n    x_min, x_max = int(xs.min()), int(xs.max())\n    y_min, y_max = int(ys.min()), int(ys.max())\n    box_w, box_h = max(1, x_max - x_min + 1), max(1, y_max - y_min + 1)\n    pad_x = int(round(box_w * padding_fraction))\n    pad_y = int(round(box_h * padding_fraction))\n    x0, y0 = max(0, x_min - pad_x), max(0, y_min - pad_y)\n    x1, y1 = min(w, x_max + pad_x + 1), min(h, y_max + pad_y + 1)\n\n    if (x1 - x0) < 8 or (y1 - y0) < 8:\n        return image_pil.copy(), (0, 0, w, h), True\n\n    return image_pil.crop((x0, y0, x1, y1)), (x0, y0, x1, y1), False\n\n\ndef hard_overlap(truth, prediction):\n    truth = np.asarray(truth, dtype=bool)\n    prediction = np.asarray(prediction, dtype=bool)\n    if truth.shape != prediction.shape:\n        raise ValueError(f"Mask shapes differ: {truth.shape} and {prediction.shape}")\n    intersection = int(np.logical_and(truth, prediction).sum())\n    total = int(truth.sum()) + int(prediction.sum())\n    union = int(np.logical_or(truth, prediction).sum())\n    return {\n        "dice": (2.0 * intersection / total) if total else 1.0,\n        "iou": (intersection / union) if union else 1.0,\n    }\n\n\ndef load_pair(segmenter_path, classifier_path):\n    seg = tf.keras.models.load_model(str(segmenter_path), compile=False)\n    cls = tf.keras.models.load_model(str(classifier_path), compile=False)\n    if tuple(seg.input_shape[1:]) != (256, 256, 3):\n        raise ValueError("Expected a 256 x 256 RGB segmenter.")\n    if len(cls.inputs) != 1 or tuple(cls.input_shape[1:]) != (224, 224, 3):\n        raise ValueError("Expected a single-input 224 x 224 RGB classifier.")\n    return seg, cls\n\n\ndef check_score(score):\n    score = float(score)\n    if not np.isfinite(score) or not 0 <= score <= 1:\n        raise ValueError(f"Invalid sigmoid model score: {score}")\n    return score\n\n\ndef infer_attention(image_path, run_dir):\n    pipeline = MelanomaPipeline.load(run_dir)\n    result = pipeline.predict_file(image_path, explain=False)\n    return {\n        "mask": np.asarray(result["mask"]) > 0,\n        "roi": result["roi"].copy(),\n        "score": check_score(result["score"]),\n        "threshold": float(pipeline.configuration["threshold"]),\n        "mask_threshold": float(pipeline.configuration["segmentation_threshold"]),\n        "mask_method": pipeline.configuration["segmentation_method"],\n        "classifier_method": pipeline.configuration["method"],\n        "roi_note": result["roi_reason"],\n    }\n\n\ndef infer_uploaded_unet(image_path, segmenter_path, classifier_path):\n    segmenter, classifier = load_pair(segmenter_path, classifier_path)\n    rgb = tf.cast(tf.io.decode_jpeg(tf.io.read_file(str(image_path)), channels=3),\n                  tf.float32)\n    seg_input = tf.image.resize(rgb, (256, 256)) / 255.0\n    probability = segmenter(seg_input[None, ...], training=False)[0, ..., 0].numpy()\n    # The supplied notebook uses > 0.5 for its ROI mask, then >= 0.5 for classification.\n    mask = probability > 0.5\n    roi_pixels = get_lesion_region(rgb, segmenter, pad_frac=0.15, mode="mask_crop")\n    score = check_score(classifier.predict(roi_pixels[None, ...], verbose=0).ravel()[0])\n    return {\n        "mask": mask,\n        "roi": Image.fromarray(np.clip(roi_pixels, 0, 255).astype(np.uint8)),\n        "score": score, "threshold": 0.5, "mask_threshold": 0.5,\n        "mask_method": "single", "classifier_method": "single",\n        "roi_note": "background masking + 15% bounding-box crop",\n    }\n\n\ndef infer_resunet(image_path, segmenter_path, classifier_path):\n    segmenter, classifier = load_pair(segmenter_path, classifier_path)\n    with Image.open(image_path) as image:\n        original = image.convert("RGB")\n    _, mask = predict_mask_for_pil(segmenter, original)\n    roi, bbox, fallback = extract_roi_from_binary_mask(original, mask, padding_fraction=0.10)\n    # Preserve the original saved quality-95 JPEG ROI preprocessing.\n    buffer = BytesIO()\n    roi.save(buffer, format="JPEG", quality=95)\n    roi_pixels = tf.io.decode_jpeg(buffer.getvalue(), channels=3)\n    roi_pixels = tf.cast(tf.image.resize(roi_pixels, (224, 224), method="bilinear"),\n                         tf.float32)\n    score = check_score(classifier.predict(roi_pixels[None, ...], verbose=0).ravel()[0])\n    return {\n        "mask": mask.astype(bool), "roi": roi.copy(),\n        "score": score, "threshold": 0.5, "mask_threshold": 0.5,\n        "mask_method": "single", "classifier_method": "single",\n        "roi_note": "full-image fallback" if fallback else "10% RGB bounding-box crop",\n    }\n\n\ndef infer_transunet(image_path, segmenter_path, classifier_path):\n    segmenter, classifier = load_pair(segmenter_path, classifier_path)\n    original = cv2.imread(str(image_path), cv2.IMREAD_COLOR)\n    if original is None:\n        raise ValueError("OpenCV could not decode the JPEG.")\n    original = cv2.cvtColor(original, cv2.COLOR_BGR2RGB)\n    # Match the supplied notebook\'s OpenCV resize and normalisation exactly.\n    seg_pixels = cv2.resize(original, (256, 256)).astype(np.float32) / 255.0\n    probability = segmenter.predict(seg_pixels[None, ...], verbose=0)[0, ..., 0]\n    mask = probability >= 0.5\n    masked_pixels = seg_pixels * mask[..., None].astype(np.float32)\n    roi_pixels = tf.image.resize(masked_pixels, (224, 224)).numpy() * 255.0\n    score = check_score(classifier.predict(roi_pixels[None, ...], verbose=0).ravel()[0])\n    return {\n        "mask": mask,\n        "roi": Image.fromarray(np.clip(roi_pixels, 0, 255).astype(np.uint8)),\n        "score": score, "threshold": 0.5, "mask_threshold": 0.5,\n        "mask_method": "single", "classifier_method": "single",\n        "roi_note": "256-pixel background-masked full frame, resized to 224",\n    }\n\n\ndef comparison_outputs(image_path, mask_path, output_dir, models, true_label="Benign"):\n    output_dir = Path(output_dir)\n    output_dir.mkdir(parents=True, exist_ok=True)\n    with Image.open(mask_path) as image:\n        # One explicitly shared reference and hard Dice/IoU formula for this figure.\n        truth = np.asarray(image.convert("L").resize((256, 256),\n                           Image.Resampling.NEAREST)) > 127\n    outputs = []\n    rows = []\n    order = ["Attention U-Net", "U-Net (uploaded notebook)", "ResUNet", "TransUNet"]\n    for name in order:\n        tf.keras.backend.clear_session()\n        gc.collect()\n        spec = models[name]\n        print(f"Running {name} on {Path(image_path).name}...")\n        if name == "Attention U-Net":\n            result = infer_attention(image_path, spec["run_dir"])\n        else:\n            function = {\n                "U-Net (uploaded notebook)": infer_uploaded_unet,\n                "ResUNet": infer_resunet,\n                "TransUNet": infer_transunet,\n            }[name]\n            result = function(image_path, spec["segmenter"], spec["classifier"])\n        metrics = hard_overlap(truth, result["mask"])\n        label = "Melanoma" if result["score"] >= result["threshold"] else "Benign"\n        result.update(name=name, prediction=label, **metrics)\n        slug = {"Attention U-Net": "attention", "U-Net (uploaded notebook)": "unet",\n                "ResUNet": "resunet", "TransUNet": "transunet"}[name]\n        Image.fromarray(result["mask"].astype(np.uint8) * 255).save(output_dir / f"{slug}_mask.png")\n        result["roi"].save(output_dir / f"{slug}_roi.png")\n        outputs.append(result)\n        rows.append({\n            "Pipeline": name, "Image ID": Path(image_path).stem, "True label": true_label,\n            "Prediction": label, "Model score": result["score"],\n            "Decision threshold": result["threshold"], "Hard Dice (this image)": metrics["dice"],\n            "Hard IoU (this image)": metrics["iou"], "Mask threshold": result["mask_threshold"],\n            "Mask inference": result["mask_method"],\n            "Classifier inference": result["classifier_method"], "ROI treatment": result["roi_note"],\n        })\n        tf.keras.backend.clear_session()\n        gc.collect()\n    table = pd.DataFrame(rows)\n    table.to_csv(output_dir / "single_image_comparison.csv", index=False)\n    return truth, outputs, table\n\n\ndef save_comparison_figures(image_path, truth, outputs, output_dir):\n    import matplotlib.pyplot as plt\n    output_dir = Path(output_dir)\n    with Image.open(image_path) as image:\n        original = image.convert("RGB").copy()\n    display_rgb = np.asarray(original.resize((256, 256), Image.Resampling.BILINEAR))\n    fig, axes = plt.subplots(1, 6, figsize=(19.5, 5.2), facecolor="white")\n    axes[0].imshow(display_rgb)\n    axes[0].set_title("Original image", fontsize=17)\n    axes[1].imshow(truth, cmap="gray", vmin=0, vmax=1)\n    axes[1].set_title("Expert mask", fontsize=17)\n    for ax, result in zip(axes[2:], outputs):\n        name = "U-Net*" if result["name"].startswith("U-Net (") else result["name"]\n        ax.imshow(result["mask"], cmap="gray", vmin=0, vmax=1)\n        ax.set_title(name, fontsize=17, fontweight="bold")\n        ax.text(0.5, -0.08,\n                f\'Dice {result["dice"]:.3f}\\n{result["prediction"]}\\n\'\n                f\'Score {result["score"]:.3f} / cutoff {result["threshold"]:.3f}\',\n                ha="center", va="top", transform=ax.transAxes, fontsize=13)\n    for ax in axes:\n        ax.axis("off")\n    fig.suptitle(f"Best Attention U-Net segmentation example: {Path(image_path).stem} (Benign)", fontsize=23,\n                 fontweight="bold", y=0.97)\n    fig.subplots_adjust(left=0.015, right=0.985, top=0.80, bottom=0.27, wspace=0.12)\n    fig.text(0.5, 0.06,\n             "Selected best Attention segmentation case. Peer weights may be newly trained. "\n             "*The uploaded U-Net++ notebook implements a standard U-Net.",\n             ha="center", fontsize=12)\n    fig.savefig(output_dir / "Slide1_Shared_Image_Comparison.png", dpi=200,\n                facecolor="white", bbox_inches="tight")\n    plt.show()\n\n    fig, axes = plt.subplots(4, 3, figsize=(10, 14), facecolor="white")\n    for i, result in enumerate(outputs):\n        axes[i, 0].imshow(result["mask"], cmap="gray", vmin=0, vmax=1)\n        axes[i, 1].imshow(display_rgb)\n        if result["mask"].any() and not result["mask"].all():\n            axes[i, 1].contour(result["mask"].astype(float), levels=[0.5],\n                               colors=["#00dd77"], linewidths=1.2)\n        axes[i, 2].imshow(result["roi"])\n        axes[i, 0].set_title(f\'{result["name"]}\\nDice {result["dice"]:.4f}\', fontsize=13)\n        axes[i, 1].set_title("Predicted boundary", fontsize=13)\n        axes[i, 2].set_title(f\'Classifier ROI\\nPrediction: {result["prediction"]}\', fontsize=13)\n    for ax in axes.flat:\n        ax.axis("off")\n    fig.suptitle(f"Pipeline details for {Path(image_path).stem} (true label: Benign)", fontsize=16)\n    fig.tight_layout(rect=(0, 0, 1, 0.97))\n    fig.savefig(output_dir / "Backup_Masks_Boundaries_and_ROIs.png", dpi=180,\n                facecolor="white", bbox_inches="tight")\n    plt.close(fig)\n', 'drive_notebook_runner.py': '"""Train missing peer pipelines from their supplied notebooks; preserve Attention weights."""\nimport ast\nimport gc\nimport hashlib\nimport json\nimport os\nimport random\nimport shutil\nimport time\nimport zipfile\nfrom contextlib import contextmanager\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nimport requests\nimport tensorflow as tf\nfrom PIL import Image\nfrom IPython.display import display\n\n\ndef code_of(nb, index):\n    cell = nb[\'cells\'][index]\n    if cell[\'cell_type\'] != \'code\':\n        raise ValueError(f\'Notebook version differs: cell {index} must contain code.\')\n    source = \'\'.join(cell[\'source\'])\n    # Keep Colab\'s installed framework, rather than running the old pip-install line.\n    source = \'\\n\'.join(line for line in source.splitlines()\n                       if not line.lstrip().startswith((\'!pip \', \'%pip \')))\n    return source\n\n\ndef require_marker(nb, index, marker):\n    if marker not in code_of(nb, index):\n        raise ValueError(f\'Notebook version differs at cell {index}: expected {marker!r}.\')\n\n\ndef locate_notebook(drive_root, prefix, exact_names):\n    direct = [drive_root / name for name in exact_names if (drive_root / name).is_file()]\n    if direct:\n        path = direct[0]\n    else:\n        matches = sorted(path for path in drive_root.rglob(\'*.ipynb\')\n                         if path.name.startswith(prefix)\n                         and not path.name.startswith(\'._\'))\n        if len(matches) != 1:\n            raise FileNotFoundError(\n                f\'Expected one {prefix} notebook in My Drive. Found {len(matches)}.\\n\'\n                + \'\\n\'.join(str(path) for path in matches)\n                + \'\\nKeep one intended copy at the top level of My Drive and rerun.\')\n        path = matches[0]\n    nb = json.loads(path.read_text())\n    digest = hashlib.sha256(\'\\n\'.join(\n        \'\'.join(cell.get(\'source\', [])) for cell in nb[\'cells\']\n        if cell[\'cell_type\'] == \'code\').encode()).hexdigest()[:12]\n    return {\'path\': path, \'nb\': nb, \'hash\': digest}\n\n\ndef locate_attention_run(drive_root):\n    preferred = drive_root / \'Melanoma_ISIC2016_Final_Candidate/20261007T163637_386595Z\'\n    def complete(path):\n        return all((path / relative).is_file() for relative in (\n            \'models/attention_unet_best.keras\', \'models/selected_final_classifier.keras\',\n            \'tables/selected_configuration.json\'))\n    if complete(preferred):\n        return preferred\n    matches = []\n    for folder_name in (\'Melanoma_Attention_UNet_Reporting_and_App\', \'Melanoma_ISIC2016_Final_Candidate\'):\n        root = drive_root / folder_name\n        if not root.is_dir():\n            continue\n        for config_path in root.rglob(\'selected_configuration.json\'):\n            run = config_path.parent.parent\n            if complete(run):\n                cfg = json.loads(config_path.read_text())\n                if (cfg.get(\'view\') == \'dual\' and cfg.get(\'method\') == \'single\'\n                        and cfg.get(\'segmentation_method\') == \'4-flip average\'\n                        and abs(float(cfg.get(\'threshold\', -1)) - .3088) < .001):\n                    matches.append(run)\n    if not matches:\n        raise FileNotFoundError(\n            \'Your frozen Attention U-Net models were not found in your Drive.\\n\'\n            \'The three peer notebooks cannot recreate your original Attention weights.\\n\'\n            \'Restore your original Attention software ZIP or saved run before continuing.\')\n    return sorted(matches, key=lambda path: path.name)[-1]\n\n\ndef run_cell(ns, nb, index, source=None):\n    source = code_of(nb, index) if source is None else source\n    # TensorFlow AutoGraph must be able to inspect augmentation/dataset functions.\n    # Store the extracted source as a real module file instead of an anonymous exec string.\n    source_folder = Path(__file__).parent / \'executed_source\'\n    source_folder.mkdir(parents=True, exist_ok=True)\n    digest = hashlib.sha256(source.encode()).hexdigest()[:12]\n    path = source_folder / f\'cell_{index}_{digest}.py\'\n    path.write_text(source + \'\\n\', encoding=\'utf-8\')\n    try:\n        exec(compile(source, str(path), \'exec\'), ns)\n    except Exception as exc:\n        raise RuntimeError(f\'{ns["_pipeline"]}: source cell {index} failed: {exc}\') from exc\n\n\ndef functions_only(source, names):\n    tree = ast.parse(source)\n    found = {node.name: node for node in tree.body if isinstance(node, ast.FunctionDef)}\n    pieces = []\n    for name in names:\n        if name not in found:\n            raise ValueError(\'Missing source function: \' + name)\n        pieces.append(ast.get_source_segment(source, found[name]))\n    return \'\\n\\n\'.join(pieces)\n\n\n@contextmanager\ndef in_folder(path):\n    path.mkdir(parents=True, exist_ok=True)\n    previous = Path.cwd()\n    os.chdir(path)\n    try:\n        yield\n    finally:\n        os.chdir(previous)\n\n\ndef free_namespace(ns):\n    ns.clear()\n    tf.keras.backend.clear_session()\n    gc.collect()\n\n\ndef download(url, path):\n    if path.is_file() and zipfile.is_zipfile(path):\n        return\n    print(\'Downloading\', path.name, flush=True)\n    temp = path.with_suffix(path.suffix + \'.partial\')\n    with requests.get(url, stream=True, timeout=(30, 180)) as response:\n        response.raise_for_status()\n        with temp.open(\'wb\') as handle:\n            for chunk in response.iter_content(8 * 1024 * 1024):\n                if chunk:\n                    handle.write(chunk)\n    if not zipfile.is_zipfile(temp):\n        raise ValueError(\'Invalid dataset ZIP: \' + path.name)\n    temp.replace(path)\n\n\ndef prepare_shared_dataset(runtime, embedded_case_dir):\n    root = runtime / \'isic2016\'\n    downloads = runtime / \'downloads\'\n    root.mkdir(parents=True, exist_ok=True)\n    downloads.mkdir(exist_ok=True)\n    base = \'https://isic-archive.s3.amazonaws.com/challenges/2016/\'\n    for split, expected in [(\'Training\', 900), (\'Test\', 379)]:\n        archive_path = downloads / f\'ISBI2016_ISIC_Part3B_{split}_Data.zip\'\n        download(base + archive_path.name, archive_path)\n        image_dir = root / f\'ISBI2016_ISIC_Part1_{split}_Data\'\n        mask_dir = root / f\'ISBI2016_ISIC_Part1_{split}_GroundTruth\'\n        image_dir.mkdir(exist_ok=True)\n        mask_dir.mkdir(exist_ok=True)\n        with zipfile.ZipFile(archive_path) as archive:\n            names = {Path(name).name: name for name in archive.namelist()\n                     if not name.endswith(\'/\') and Path(name).name.startswith(\'ISIC_\')}\n            for name, member in names.items():\n                destination = image_dir / name if name.lower().endswith(\'.jpg\') else mask_dir / name\n                if name.lower().endswith(\'.jpg\') or name.endswith(\'_Segmentation.png\'):\n                    if not destination.is_file():\n                        with archive.open(member) as source, destination.open(\'wb\') as target:\n                            shutil.copyfileobj(source, target)\n        assert len(list(image_dir.glob(\'ISIC_*.jpg\'))) == expected\n        assert len(list(mask_dir.glob(\'ISIC_*_Segmentation.png\'))) == expected\n        alias = root / f\'ISBI2016_ISIC_Part3_{split}_Data\'\n        if not alias.exists():\n            alias.symlink_to(image_dir, target_is_directory=True)\n        source_labels = embedded_case_dir / f\'ISBI2016_ISIC_Part3B_{split}_GroundTruth.csv\'\n        shutil.copy2(source_labels, root / f\'ISBI2016_ISIC_Part3_{split}_GroundTruth.csv\')\n    print(\'Shared dataset ready: 900 official training and 379 official test images.\', flush=True)\n    return root, downloads / \'ISBI2016_ISIC_Part3B_Training_Data.zip\'\n\n\ndef new_namespace(name):\n    return {\'__name__\': \'__main__\', \'_pipeline\': name, \'display\': lambda *args, **kwargs: None}\n\n\ndef train_uploaded_unet(record, folder, data_dir):\n    nb = record[\'nb\']\n    ns = new_namespace(\'U-Net from uploaded notebook\')\n    folder.mkdir(parents=True, exist_ok=True)\n    seg_file, cls_file = folder / \'selected_segmenter.keras\', folder / \'selected_classifier.keras\'\n    print(\'\\nU-Net: training from the supplied notebook. This file implements standard U-Net.\', flush=True)\n    try:\n        with in_folder(folder):\n            run_cell(ns, nb, 1)\n            ns[\'DATA_DIR\'] = str(data_dir)\n            for index in (5, 7, 8):\n                run_cell(ns, nb, index)\n            if seg_file.is_file():\n                ns[\'unet\'] = tf.keras.models.load_model(seg_file, compile=False)\n                print(\'Reusing completed U-Net segmentation stage.\', flush=True)\n            else:\n                run_cell(ns, nb, 9)\n                ns[\'unet\'].save(seg_file)\n            run_cell(ns, nb, 14)\n            roi_cache = folder / \'roi_cache\'\n            roi_cache.mkdir(exist_ok=True)\n            ns[\'ROI_CACHE_DIR\'] = roi_cache\n            source = code_of(nb, 15).replace(\n                \'cache_path = os.path.join(DATA_DIR, cache_name)\',\n                \'cache_path = os.path.join(str(ROI_CACHE_DIR), cache_name)\')\n            if source == code_of(nb, 15):\n                raise ValueError(\'The U-Net ROI cache code differs from the supplied version.\')\n            for index, replacement in [(15, source), (16, None), (18, None), (22, None)]:\n                run_cell(ns, nb, index, source=replacement)\n            ns[\'cls_model\'].save(cls_file)\n        return seg_file, cls_file\n    finally:\n        free_namespace(ns)\n\n\ndef train_resunet(record, folder, data_dir, archive_path, case_dir, runtime):\n    nb = record[\'nb\']\n    ns = new_namespace(\'ResUNet\')\n    folder.mkdir(parents=True, exist_ok=True)\n    seg_file, cls_file = folder / \'selected_segmenter.keras\', folder / \'selected_classifier.keras\'\n    print(\'\\nResUNet: training its original 160/40 development split.\', flush=True)\n    try:\n        run_cell(ns, nb, 1)\n        settings = code_of(nb, 3).split(\'# 2. STORAGE\')[0]\n        run_cell(ns, nb, 3, settings)\n        local = runtime / \'resunet\'\n        assignments = {\n            \'RUNTIME_ROOT\': local, \'DOWNLOAD_DIR\': runtime / \'downloads\',\n            \'DATA_DIR\': local / \'selected_300\', \'ROI_DIR\': local / \'roi_images\',\n            \'AUGMENTED_ROI_DIR\': local / \'targeted_augmented_roi_images\',\n            \'PROJECT_DIR\': folder, \'MODELS_DIR\': folder / \'models\',\n            \'RESULTS_DIR\': folder / \'results\', \'FIGURES_DIR\': folder / \'figures\',\n            \'SPLITS_DIR\': folder / \'splits\', \'DATA_ZIP\': archive_path,\n            \'LABEL_CSV\': case_dir / \'ISBI2016_ISIC_Part3B_Training_GroundTruth.csv\',\n        }\n        ns.update(assignments)\n        for key, path in assignments.items():\n            if key not in (\'DATA_ZIP\', \'LABEL_CSV\'):\n                path.mkdir(parents=True, exist_ok=True)\n        run_cell(ns, nb, 5)\n        # Use the already extracted shared files. Do not let the original cell delete the shared ZIP.\n        for name in (\'train_df\', \'val_df\', \'model_df\', \'test_df\', \'selected_cases\'):\n            frame = ns[name].copy()\n            frame[\'image_path\'] = frame[\'image_id\'].map(\n                lambda image_id: str(data_dir / \'ISBI2016_ISIC_Part1_Training_Data\' / f\'{image_id}.jpg\'))\n            frame[\'mask_path\'] = frame[\'image_id\'].map(\n                lambda image_id: str(data_dir / \'ISBI2016_ISIC_Part1_Training_GroundTruth\' / f\'{image_id}_Segmentation.png\'))\n            ns[name] = frame\n        run_cell(ns, nb, 9, functions_only(code_of(nb, 9), [\'load_rgb\', \'load_mask\']))\n        # Dataset functions are at the end of source cell 9, before the loss/architecture cell.\n        data_source = code_of(nb, 9)\n        start = data_source.index(\'def decode_segmentation_pair\')\n        run_cell(ns, nb, 9, data_source[start:])\n        run_cell(ns, nb, 11)\n        if seg_file.is_file():\n            ns[\'selected_segmenter\'] = tf.keras.models.load_model(seg_file, compile=False)\n            print(\'Reusing completed ResUNet segmentation stage.\', flush=True)\n        else:\n            run_cell(ns, nb, 13)\n            ns[\'selected_segmenter\'].save(seg_file)\n        run_cell(ns, nb, 15, functions_only(code_of(nb, 15), [\'predict_mask_for_pil\']))\n        source = code_of(nb, 17)\n        marker = \'# Crop the ORIGINAL RGB image; do not classify masks\'\n        if marker not in source:\n            raise ValueError(\'ResUNet ROI/training cell differs from the uploaded version.\')\n        run_cell(ns, nb, 17, source[source.index(marker):])\n        run_cell(ns, nb, 19)\n        ns[\'final_classifier\'].save(cls_file)\n        return seg_file, cls_file\n    finally:\n        free_namespace(ns)\n\n\ndef train_transunet(record, folder, data_dir):\n    nb = record[\'nb\']\n    ns = new_namespace(\'TransUNet\')\n    folder.mkdir(parents=True, exist_ok=True)\n    seg_file, cls_file = folder / \'selected_segmenter.keras\', folder / \'selected_classifier.keras\'\n    print(\'\\nTransUNet: training its original pooled-data 895/128 development split.\', flush=True)\n    try:\n        with in_folder(folder):\n            run_cell(ns, nb, 2)\n            ns[\'DATA_DIR\'] = str(data_dir)\n            for index in (9, 10, 11, 16, 21):\n                run_cell(ns, nb, index)\n            ns.pop(\'images\', None); ns.pop(\'masks\', None); ns.pop(\'labels\', None)\n            ns.pop(\'X_temp\', None); ns.pop(\'y_temp\', None); ns.pop(\'t_temp\', None)\n            gc.collect()\n            if seg_file.is_file():\n                ns[\'transunet\'] = tf.keras.models.load_model(seg_file, compile=False)\n                print(\'Reusing completed TransUNet segmentation stage.\', flush=True)\n            else:\n                for index in (23, 25, 26, 27, 28):\n                    run_cell(ns, nb, index)\n                ns[\'transunet\'].save(seg_file)\n            # Keep the final corrected augmented classifier only; skip the earlier failed/scaling experiments.\n            for index in (84, 85, 86):\n                run_cell(ns, nb, index)\n            for name in (\'X_train\', \'X_val\', \'X_test\', \'y_train\', \'y_val\', \'y_test\',\n                         \'train_masks_pred\', \'val_masks_pred\', \'test_masks_pred\',\n                         \'train_masks_binary\', \'val_masks_binary\', \'test_masks_binary\',\n                         \'X_train_roi\', \'X_val_roi\', \'X_test_roi\', \'X_test_cls\',\n                         \'transunet\', \'history\', \'callbacks\'):\n                ns.pop(name, None)\n            tf.keras.backend.clear_session(); gc.collect()\n            for index in (87, 88, 89, 90):\n                run_cell(ns, nb, index)\n            source = code_of(nb, 91).replace(\n                \'"/content/drive/MyDrive/43022/EfficientNetB0_Augmented.keras"\',\n                repr(str(folder / \'best_augmented_classifier.keras\')))\n            if source == code_of(nb, 91):\n                raise ValueError(\'TransUNet classifier checkpoint path differs from the supplied notebook.\')\n            run_cell(ns, nb, 91, source)\n            ns[\'classifier_aug\'].save(cls_file)\n        return seg_file, cls_file\n    finally:\n        free_namespace(ns)\n\n\ndef obtain_peer_models(records, output_root, drive_root, case_dir, runtime):\n    originals = {\n        \'U-Net (uploaded notebook)\': (\n            drive_root / \'Melanoma_Comparison_Models/unet_segmenter.keras\',\n            drive_root / \'Melanoma_Comparison_Models/unet_classifier.keras\'),\n        \'ResUNet\': (\n            drive_root / \'Melanoma_AI_ResUNet_Targeted_Malignant_Aug_Results/models/resunet_best.keras\',\n            drive_root / \'Melanoma_AI_ResUNet_Targeted_Malignant_Aug_Results/models/efficientnet_b0_final.keras\'),\n        \'TransUNet\': (\n            drive_root / \'43022/TransUNet_ISIC2016.keras\',\n            drive_root / \'43022/EfficientNetB0_Augmented_Final.keras\'),\n    }\n    results, provenance, missing = {}, [], []\n    for name, record in records.items():\n        folder = output_root / \'trained_models\' / record[\'hash\'] / name.split()[0].lower()\n        cached = (folder / \'selected_segmenter.keras\', folder / \'selected_classifier.keras\')\n        pair = cached if all(path.is_file() for path in cached) else originals[name]\n        if all(path.is_file() for path in pair):\n            results[name] = {\'segmenter\': pair[0], \'classifier\': pair[1]}\n            provenance.append({\'Pipeline\': name, \'Weights\': \'reused saved checkpoints\',\n                               \'Notebook\': str(record[\'path\']), \'Code hash\': record[\'hash\']})\n            print(\'Reusing saved models:\', name, flush=True)\n        else:\n            missing.append((name, record, folder))\n    if missing:\n        if not tf.config.list_physical_devices(\'GPU\'):\n            raise RuntimeError(\'Missing peer models need training. Select Runtime > Change runtime type > T4 GPU, then rerun this cell.\')\n        print(\'\\nMissing trained models:\', \', \'.join(name for name, _, _ in missing), flush=True)\n        print(\'Training can take hours. These will be NEW RUNS, not the old reported results.\', flush=True)\n        data_dir, archive = prepare_shared_dataset(runtime, case_dir)\n        for name, record, folder in missing:\n            tf.keras.backend.clear_session(); gc.collect()\n            if name.startswith(\'U-Net\'):\n                pair = train_uploaded_unet(record, folder, data_dir)\n            elif name == \'ResUNet\':\n                pair = train_resunet(record, folder, data_dir, archive, case_dir, runtime)\n            else:\n                pair = train_transunet(record, folder, data_dir)\n            results[name] = {\'segmenter\': pair[0], \'classifier\': pair[1]}\n            provenance.append({\'Pipeline\': name, \'Weights\': \'NEW training run from notebook code\',\n                               \'Notebook\': str(record[\'path\']), \'Code hash\': record[\'hash\']})\n            pd.DataFrame(provenance).to_csv(output_root / \'model_provenance.csv\', index=False)\n    pd.DataFrame(provenance).to_csv(output_root / \'model_provenance.csv\', index=False)\n    return results, provenance\n'}

for name, text in MODULES.items():
    (MODULE_DIR / name).write_text(text, encoding='utf-8')
sys.path.insert(0, str(MODULE_DIR))
importlib.invalidate_caches()
for module_name in ('drive_notebook_runner', 'bestcase_comparison_helpers', 'attention_pipeline_core'):
    sys.modules.pop(module_name, None)
from drive_notebook_runner import (
    locate_notebook, locate_attention_run, require_marker, obtain_peer_models,
)
from bestcase_comparison_helpers import comparison_outputs, save_comparison_figures

# Locate your original Attention models before starting any expensive peer training.
attention_run = locate_attention_run(DRIVE_ROOT)
print('Using saved Attention U-Net:', attention_run)

records = {
    'U-Net (uploaded notebook)': locate_notebook(
        DRIVE_ROOT, 'UNet++ Final Code', ['UNet++ Final Code(1).ipynb', 'UNet++ Final Code.ipynb']),
    'ResUNet': locate_notebook(
        DRIVE_ROOT, 'ResUNet_TARGETED_MELANOMA_AUGMENTATION_READY', [
            'ResUNet_TARGETED_MELANOMA_AUGMENTATION_READY (1) (1)(1).ipynb',
            'ResUNet_TARGETED_MELANOMA_AUGMENTATION_READY (1) (1).ipynb']),
    'TransUNet': locate_notebook(
        DRIVE_ROOT, 'ISIC_2016_TransUNet', ['ISIC_2016_TransUNet.ipynb']),
}
require_marker(records['U-Net (uploaded notebook)']['nb'], 8, 'def build_unet_segmenter')
require_marker(records['U-Net (uploaded notebook)']['nb'], 22, 'history_phase1 = cls_model.fit')
require_marker(records['ResUNet']['nb'], 11, 'def build_resunet')
require_marker(records['ResUNet']['nb'], 17, '# Crop the ORIGINAL RGB image; do not classify masks')
require_marker(records['TransUNet']['nb'], 23, 'def build_transunet')
require_marker(records['TransUNet']['nb'], 91, 'history_aug = classifier_aug.fit')
for name, record in records.items():
    print('Found notebook:', name, '|', record['path'].name)

print('Selected image:', IMAGE_ID, '| Actual diagnosis: Benign')
print('Historical Attention segmentation Dice: approximately 0.9876.')
print('This is a selected best-case illustration, not an overall model ranking.')

models, provenance = obtain_peer_models(
    records, OUTPUT_ROOT, DRIVE_ROOT, CASE_DIR, RUNTIME)
models['Attention U-Net'] = {'run_dir': attention_run}

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')
OUTPUT_DIR = OUTPUT_ROOT / 'comparisons' / RUN_ID
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
truth, outputs, table = comparison_outputs(
    IMAGE_PATH, MASK_PATH, OUTPUT_DIR, models, true_label='Benign')
display(table.round(4))
save_comparison_figures(IMAGE_PATH, truth, outputs, OUTPUT_DIR)

pd.DataFrame(provenance).to_csv(OUTPUT_DIR / 'model_provenance.csv', index=False)
manifest = {
    'image_id': IMAGE_ID, 'true_label': 'Benign',
    'selection': 'Recorded best Attention U-Net test segmentation case',
    'historical_attention_dice': 0.9876,
    'source': 'https://isic-archive.s3.amazonaws.com/challenges/2016/ISBI2016_ISIC_Part3B_Test_Data.zip',
    'image_sha256': EMBEDDED_FILES[f'{IMAGE_ID}.jpg']['sha256'],
    'mask_sha256': EMBEDDED_FILES[f'{IMAGE_ID}_Segmentation.png']['sha256'],
    'attention_saved_run': str(attention_run), 'peer_model_provenance': provenance,
    'limitations': [
        'Selected best-case example for Attention segmentation',
        'Benign image; this case does not measure melanoma sensitivity',
        'Different training/validation/test splits and ROI processing',
        'Retrained peers are new runs and are not the models behind the old reported metrics',
        'The supplied U-Net++ notebook implements standard U-Net',
        'The Attention official test set was previously examined',
    ],
}
(OUTPUT_DIR / 'comparison_provenance.json').write_text(json.dumps(manifest, indent=2))
package_path = OUTPUT_DIR / f'{IMAGE_ID}_BestCase_Comparison.zip'
with zipfile.ZipFile(package_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(OUTPUT_DIR.iterdir()):
        if path.is_file() and path != package_path:
            archive.write(path, arcname=path.name)
    archive.write(IMAGE_PATH, arcname=IMAGE_PATH.name)
    archive.write(MASK_PATH, arcname=MASK_PATH.name)
print('Finished. Slide figure:', OUTPUT_DIR / 'Slide1_Shared_Image_Comparison.png')
print('New peer checkpoints are saved for reuse; the original notebooks were not edited.')
files.download(str(package_path))
